# Predicting Viral AI Tweets — SWA2124 Social & Web Analytics

**Group:** Cheah Choon Keat (leader), Brandon Wong Kai Ian, Jehuda Rhema Chang, Nihal, Palani

This single notebook contains the **entire project code** and reproduces **every table and
every figure** in the report. It:
1. installs the libraries,
2. downloads the open **tweets_ai** dataset from **Harvard Dataverse** (DOI `10.7910/DVN/NHLEJL`),
3. writes the three project scripts to disk and runs the full pipeline, then
4. displays every result table (Tables III–VIII) and every figure (Figs 1–12 + appendix) inline.

> Runtime ≈ 15–25 min on a free Colab CPU (the dataset is 380 MB and the tuning step is heavy).
> `Runtime → Run all`. Every random seed is fixed at 42.


## 1. Install libraries

In [ ]:
!pip -q install vaderSentiment imbalanced-learn xgboost >/dev/null
print("libraries ready")

## 2. Download the dataset from Harvard Dataverse
Dataverse blocks requests with no browser `User-Agent`, so we send one.

In [ ]:
import os, urllib.request, shutil
URL = "https://dataverse.harvard.edu/api/access/datafile/11812857"  # tweets_ai.csv
PATH = "tweets_ai.csv"
if not os.path.exists(PATH):
    print("downloading ~380 MB (about a minute) ...")
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(PATH, "wb") as f:
        shutil.copyfileobj(r, f)
print("size (MB):", round(os.path.getsize(PATH)/1e6, 1))

## 3. The full project code (written to disk)
The next three cells contain every line of the project: data preparation, the machine-learning study (11 models, tuning, all tables) and all the figures.

In [ ]:
%%writefile swa2124_analysis.py
# SWA2124 Social and Web Analytics - Final Assessment
# STEP 1 of 3 : data preparation + Power BI exports
# (STEP 2 = swa2124_models.py, STEP 3 = swa2124_modelfigs.py)
#
# Dataset: tweets_ai.csv (Harvard Dataverse, DOI 10.7910/DVN/NHLEJL) -
# 893,076 tweets about artificial intelligence, 2017-2021.
#
# Usage:
#   python swa2124_analysis.py preprocess   -> clean data + VADER sentiment + features
#                                              (writes outputs/processed_tweets.csv)
#   python swa2124_analysis.py exports      -> aggregate CSVs for the Power BI dashboard
#   python swa2124_analysis.py all          -> both of the above

import os
import re
import sys
import time

import numpy as np
import pandas as pd

BASE = os.path.dirname(os.path.abspath(__file__))
RAW = os.path.join(BASE, "tweets_ai.csv")
OUT = os.path.join(BASE, "outputs")
PROCESSED = os.path.join(OUT, "processed_tweets.csv")
os.makedirs(OUT, exist_ok=True)

RNG = 42  # fixed seed so results are reproducible


def count_list_items(s):
    # columns like hashtags/urls are stored as string lists e.g. "['ai', 'ml']"
    if not isinstance(s, str) or s in ("[]", "", "NA"):
        return 0
    return s.count(",") + 1


URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")


def clean_text(t):
    t = URL_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip().lower()


def preprocess():
    """Clean the raw tweets, engineer features and score VADER sentiment."""
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

    t0 = time.time()
    usecols = ["id", "date", "time", "tweet", "language", "urls", "photos",
               "replies_count", "retweets_count", "likes_count", "hashtags", "video"]
    df = pd.read_csv(RAW, usecols=usecols,
                     dtype={"id": str, "video": str}, low_memory=False)
    print("raw rows:", len(df))

    # keep English tweets only, drop duplicates and empty text
    df = df[df["language"] == "en"].copy()
    df = df.drop_duplicates(subset="id")
    df = df[df["tweet"].notna() & (df["tweet"].str.strip() != "")]
    print("after english filter + dedupe:", len(df))

    for c in ["replies_count", "retweets_count", "likes_count"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    # total engagement, and the binary "any engagement" flag used for descriptives
    df["engagement_total"] = df["likes_count"] + df["retweets_count"] + df["replies_count"]
    df["engaged"] = (df["engagement_total"] > 0).astype(int)

    # time features
    dt = pd.to_datetime(df["date"], errors="coerce")
    df = df[dt.notna()]
    dt = dt[dt.notna()]
    df["year"] = dt.dt.year
    df["month"] = dt.dt.to_period("M").astype(str)
    df["dayofweek"] = dt.dt.dayofweek  # 0 = Monday
    df["hour"] = pd.to_numeric(df["time"].str[:2], errors="coerce").fillna(12).astype(int)
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

    # content features
    df["text_len"] = df["tweet"].str.len()
    df["word_count"] = df["tweet"].str.split().str.len()
    df["hashtag_count"] = df["hashtags"].apply(count_list_items)
    df["mention_count"] = df["tweet"].str.count("@")
    df["has_url"] = df["urls"].apply(lambda s: 1 if count_list_items(s) > 0 else 0)
    df["has_photo"] = df["photos"].apply(lambda s: 1 if count_list_items(s) > 0 else 0)
    df["has_video"] = (df["video"] == "1").astype(int)
    df["exclam_count"] = df["tweet"].str.count("!")
    df["question_mark"] = (df["tweet"].str.count(r"\?") > 0).astype(int)

    df["clean_text"] = df["tweet"].apply(clean_text)
    df = df[df["clean_text"].str.len() > 0]

    # VADER sentiment on the cleaned text (this is the slow part, ~2 min)
    print("running VADER on", len(df), "tweets ...")
    analyzer = SentimentIntensityAnalyzer()
    df["vader_compound"] = [analyzer.polarity_scores(t)["compound"] for t in df["clean_text"]]
    df["sentiment"] = np.select(
        [df["vader_compound"] >= 0.05, df["vader_compound"] <= -0.05],
        ["positive", "negative"], default="neutral")

    keep = ["id", "date", "month", "year", "hour", "dayofweek", "is_weekend",
            "clean_text", "text_len", "word_count", "hashtag_count", "mention_count",
            "has_url", "has_photo", "has_video", "exclam_count", "question_mark",
            "vader_compound", "sentiment",
            "likes_count", "retweets_count", "replies_count", "engagement_total", "engaged"]
    df[keep].to_csv(PROCESSED, index=False)
    print("saved", PROCESSED, "rows:", len(df))
    print("preprocess took %.1f min" % ((time.time() - t0) / 60))


def exports():
    """Aggregate CSVs loaded by the Power BI dashboard (see INSTRUCTIONS)."""
    df = pd.read_csv(PROCESSED)

    df.groupby("month").agg(
        tweets=("id", "count"), mean_sentiment=("vader_compound", "mean"),
        engagement_rate=("engaged", "mean"), total_likes=("likes_count", "sum"),
        total_retweets=("retweets_count", "sum")).reset_index().to_csv(
        os.path.join(OUT, "pbi_monthly.csv"), index=False)

    df.groupby(["year", "sentiment"]).size().unstack(fill_value=0).reset_index().to_csv(
        os.path.join(OUT, "pbi_sentiment_by_year.csv"), index=False)

    df.groupby("sentiment").agg(
        tweets=("id", "count"), engagement_rate=("engaged", "mean"),
        mean_likes=("likes_count", "mean"), mean_retweets=("retweets_count", "mean")
    ).reset_index().to_csv(os.path.join(OUT, "pbi_engagement_by_sentiment.csv"), index=False)

    df.groupby("hour").agg(tweets=("id", "count"),
                           engagement_rate=("engaged", "mean")).reset_index().to_csv(
        os.path.join(OUT, "pbi_engagement_by_hour.csv"), index=False)

    df.groupby("dayofweek").agg(tweets=("id", "count"),
                                engagement_rate=("engaged", "mean")).reset_index().to_csv(
        os.path.join(OUT, "pbi_engagement_by_dayofweek.csv"), index=False)

    df.groupby(["has_photo", "has_video"]).agg(
        tweets=("id", "count"), engagement_rate=("engaged", "mean")).reset_index().to_csv(
        os.path.join(OUT, "pbi_engagement_by_media.csv"), index=False)

    # a manageable tweet-level sample for dashboard drill-down (has engagement_total,
    # so the Power BI "viral rate" measure engagement_total >= 6 works)
    df.sample(50000, random_state=RNG).to_csv(
        os.path.join(OUT, "pbi_tweet_sample.csv"), index=False)
    print("power bi exports saved to", OUT)


if __name__ == "__main__":
    stage = sys.argv[1] if len(sys.argv) > 1 else "all"
    if stage in ("preprocess", "all"):
        preprocess()
    if stage in ("exports", "all"):
        exports()


In [ ]:
%%writefile swa2124_models.py
# SWA2124 - Predicting viral (high-engagement) AI tweets.
#
# Full comparative modelling study used in the report:
#   - target: viral = (likes + retweets + replies) >= 6  (top ~11%, imbalanced)
#   - leakage-free pipeline: RF-ranked feature selection -> standardisation ->
#     SMOTE-Tomek resampling (train only)
#   - 10 baseline classifiers + a proposed Viral Stacking Ensemble (VSE)
#   - per-algorithm hyperparameter tuning (GridSearchCV, each with its own grid)
#   - 9 evaluation metrics, 10-fold CV, paired significance tests, ablation study
#
# Reads outputs/processed_tweets.csv (from swa2124_analysis.py preprocess) and
# writes all result tables (CSV) and score files into outputs/.

import os
import json
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import (RandomForestClassifier, StackingClassifier,
                              AdaBoostClassifier, ExtraTreesClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             matthews_corrcoef, cohen_kappa_score, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")

BASE = os.path.dirname(os.path.abspath(__file__))
OUT = os.path.join(BASE, "outputs")
PROCESSED = os.path.join(OUT, "processed_tweets.csv")
os.makedirs(OUT, exist_ok=True)

RNG = 42
VIRAL_THRESHOLD = 6
MODEL_SAMPLE = 12000
TFIDF_FEATURES = 4000
SELECT_K = 300
SVC_CAP = 9000           # cap SVM-RBF training rows (rbf cost is ~quadratic)

NUMERIC_FEATS = ["text_len", "word_count", "hashtag_count", "mention_count",
                 "has_url", "has_photo", "has_video", "exclam_count", "question_mark",
                 "hour", "dayofweek", "is_weekend", "vader_compound"]

PROPOSED = "Proposed VSE"
BASELINES = ["Logistic Regression", "Naive Bayes", "k-NN", "Decision Tree",
             "Random Forest", "Extra Trees", "AdaBoost", "SVM (RBF)", "XGBoost",
             "MLP (Neural Net)"]


def make_model(name, p=None):
    """Construct a classifier, optionally with tuned hyperparameters p."""
    p = p or {}
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=1000, random_state=RNG, **p)
    if name == "Naive Bayes":
        return GaussianNB(**p)
    if name == "k-NN":
        return KNeighborsClassifier(n_jobs=-1, **p)
    if name == "Decision Tree":
        return DecisionTreeClassifier(random_state=RNG, **p)
    if name == "Random Forest":
        return RandomForestClassifier(n_jobs=-1, random_state=RNG, **p)
    if name == "Extra Trees":
        return ExtraTreesClassifier(n_jobs=-1, random_state=RNG, **p)
    if name == "AdaBoost":
        return AdaBoostClassifier(random_state=RNG, **p)
    if name == "SVM (RBF)":
        return SVC(kernel="rbf", random_state=RNG, **p)
    if name == "XGBoost":
        return XGBClassifier(tree_method="hist", eval_metric="logloss", n_jobs=-1,
                             random_state=RNG, **p)
    if name == "MLP (Neural Net)":
        return MLPClassifier(max_iter=250, early_stopping=True, random_state=RNG, **p)
    raise ValueError(name)


# each algorithm has its own hyperparameter grid searched with 3-fold CV (scoring=F1)
GRIDS = {
    "Logistic Regression": {"C": [0.1, 1.0, 10.0], "class_weight": [None, "balanced"]},
    "Naive Bayes": {"var_smoothing": [1e-9, 1e-8, 1e-7]},
    "k-NN": {"n_neighbors": [11, 15, 25], "weights": ["uniform", "distance"]},
    "Decision Tree": {"max_depth": [10, 20, None], "min_samples_leaf": [2, 5]},
    "Random Forest": {"n_estimators": [200, 400], "max_depth": [None, 20],
                      "min_samples_leaf": [1, 2]},
    "Extra Trees": {"n_estimators": [200, 400], "min_samples_leaf": [1, 2]},
    "AdaBoost": {"n_estimators": [100, 200], "learning_rate": [0.5, 1.0]},
    "SVM (RBF)": {"C": [1.0, 2.0, 5.0], "gamma": ["scale", 0.01]},
    "XGBoost": {"n_estimators": [200, 400], "max_depth": [4, 6],
                "learning_rate": [0.05, 0.1]},
    "MLP (Neural Net)": {"hidden_layer_sizes": [(64,), (96,)], "alpha": [1e-4, 1e-3]},
}


def proposed_model(best):
    """Stacking ensemble built from the tuned LR, RF and XGB base learners."""
    lr = make_model("Logistic Regression", best.get("Logistic Regression"))
    rf = make_model("Random Forest", best.get("Random Forest"))
    xgb = make_model("XGBoost", best.get("XGBoost"))
    return StackingClassifier(
        estimators=[("lr", lr), ("rf", rf), ("xgb", xgb)],
        final_estimator=LogisticRegression(max_iter=1000, random_state=RNG),
        stack_method="predict_proba", cv=3, n_jobs=-1)


def load_sample():
    df = pd.read_csv(PROCESSED)
    df["viral"] = (df["engagement_total"] >= VIRAL_THRESHOLD).astype(int)
    samp = df.groupby("viral", group_keys=False).sample(frac=MODEL_SAMPLE / len(df),
                                                        random_state=RNG)
    samp = samp.sample(frac=1, random_state=RNG).reset_index(drop=True)
    return df, samp


def build_features(samp):
    vec = TfidfVectorizer(max_features=TFIDF_FEATURES, ngram_range=(1, 2),
                          min_df=5, stop_words="english")
    Xtext = vec.fit_transform(samp["clean_text"].fillna(""))
    Xnum = csr_matrix(samp[NUMERIC_FEATS].values.astype(float))
    X = hstack([Xtext, Xnum]).tocsr()
    names = np.array(list(vec.get_feature_names_out()) + NUMERIC_FEATS)
    return X, samp["viral"].values, names


def scores_of(clf, X):
    if hasattr(clf, "predict_proba"):
        return clf.predict_proba(X)[:, 1]
    return clf.decision_function(X)


def all_metrics(y, yhat, sc):
    return {"Accuracy": accuracy_score(y, yhat),
            "Precision": precision_score(y, yhat, zero_division=0),
            "Recall": recall_score(y, yhat), "F1": f1_score(y, yhat),
            "Macro-F1": f1_score(y, yhat, average="macro"),
            "ROC-AUC": roc_auc_score(y, sc), "PR-AUC": average_precision_score(y, sc),
            "MCC": matthews_corrcoef(y, yhat), "Kappa": cohen_kappa_score(y, yhat)}


def prep(X, y):
    """Split, RF-rank + keep top-K features, standardise (all fit on train only).
    Returns scaled and unscaled selected-feature matrices."""
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y,
                                          random_state=RNG)
    ranker = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RNG).fit(Xtr, ytr)
    idx = np.argsort(ranker.feature_importances_)[::-1][:SELECT_K]
    Xtr_raw, Xte_raw = Xtr[:, idx].toarray(), Xte[:, idx].toarray()
    sca = StandardScaler().fit(Xtr_raw)
    return (sca.transform(Xtr_raw), sca.transform(Xte_raw), ytr, yte, idx, ranker,
            Xtr_raw, Xte_raw)


SVM_TUNE_N = 5000        # rows used for the (slow) SVM grid search


def tune(X, y, names):
    """Table II (tuning) + Table III (hold-out). Each algorithm is grid-searched with
    a leakage-free imblearn pipeline (SMOTE-Tomek applied inside every CV fold), then
    the tuned model is refit on the full resampled train and scored on the held-out
    test set."""
    Xtr_s, Xte_s, ytr, yte, idx, ranker, Xtr_raw, Xte_raw = prep(X, y)
    Xtr_r, ytr_r = SMOTETomek(random_state=RNG).fit_resample(Xtr_s, ytr)
    print("train after SMOTE-Tomek:", Xtr_r.shape, "pos rate", round(ytr_r.mean(), 3))

    best_params, tuning_rows, hold_rows = {}, [], []
    roc_data, preds = {"y_true": yte}, {}

    for name in BASELINES:
        # leakage-free grid search: scaler + SMOTE-Tomek inside each CV fold
        grid = {f"clf__{k}": v for k, v in GRIDS[name].items()}
        pipe = ImbPipeline([("sc", StandardScaler()),
                            ("sm", SMOTETomek(random_state=RNG)),
                            ("clf", make_model(name))])
        Xg, yg = Xtr_raw, ytr
        if name == "SVM (RBF)" and len(yg) > SVM_TUNE_N:
            sub = np.random.RandomState(RNG).permutation(len(yg))[:SVM_TUNE_N]
            Xg, yg = Xg[sub], yg[sub]
        gs = GridSearchCV(pipe, grid, scoring="f1", cv=3, n_jobs=-1)
        gs.fit(Xg, yg)
        best = {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}
        best_params[name] = best
        # refit tuned + default models on the full resampled train, score on test
        Xfit, yfit = Xtr_r, ytr_r
        if name == "SVM (RBF)" and len(yfit) > SVC_CAP:
            s2 = np.random.RandomState(RNG).permutation(len(yfit))[:SVC_CAP]
            Xfit, yfit = Xfit[s2], yfit[s2]
        clf = make_model(name, best).fit(Xfit, yfit)
        yhat, sc = clf.predict(Xte_s), scores_of(clf, Xte_s)
        roc_data[name], preds[name] = sc, yhat
        f1_default = f1_score(yte, make_model(name).fit(Xfit, yfit).predict(Xte_s))
        m = all_metrics(yte, yhat, sc)
        hold_rows.append({"Model": name, **m})
        tuning_rows.append({"Model": name,
                            "Best parameters": ", ".join(f"{k}={v}" for k, v in best.items()),
                            "F1 (default)": round(f1_default, 4),
                            "F1 (tuned)": round(m["F1"], 4),
                            "dF1": round(m["F1"] - f1_default, 4)})
        print(f"  {name:20s} tuned F1={m['F1']:.3f} (def {f1_default:.3f})  best={best}")

    # proposed ensemble from tuned base learners
    vse = proposed_model(best_params).fit(Xtr_r, ytr_r)
    yhat, sc = vse.predict(Xte_s), scores_of(vse, Xte_s)
    roc_data[PROPOSED], preds[PROPOSED] = sc, yhat
    m = all_metrics(yte, yhat, sc)
    hold_rows.append({"Model": PROPOSED, **m})
    print(f"  {PROPOSED:20s} F1={m['F1']:.3f} PR-AUC={m['PR-AUC']:.3f} MCC={m['MCC']:.3f}")

    pd.DataFrame(hold_rows).round(4).to_csv(os.path.join(OUT, "tab_holdout.csv"), index=False)
    pd.DataFrame(tuning_rows).to_csv(os.path.join(OUT, "tab_tuning.csv"), index=False)
    pd.DataFrame(roc_data).to_csv(os.path.join(OUT, "viral_scores.csv"), index=False)
    pd.DataFrame(confusion_matrix(yte, preds[PROPOSED])).to_csv(
        os.path.join(OUT, "viral_confusion.csv"), index=False)
    imp = pd.Series(ranker.feature_importances_[idx], index=names[idx])
    imp.sort_values(ascending=False).head(15).to_csv(os.path.join(OUT, "viral_importance.csv"))
    with open(os.path.join(OUT, "best_params.json"), "w") as f:
        json.dump({k: {kk: (list(vv) if isinstance(vv, tuple) else vv)
                       for kk, vv in v.items()} for k, v in best_params.items()}, f, indent=2)
    return best_params


def cv_pipe(clf):
    return ImbPipeline([("sc", StandardScaler()),
                        ("sm", SMOTETomek(random_state=RNG)), ("clf", clf)])


def run_cv(X, y, best):
    """Tables IV & V: 10-fold CV + paired significance for RF, XGB and proposed VSE
    (all using tuned hyperparameters)."""
    ranker = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RNG).fit(X, y)
    idx = np.argsort(ranker.feature_importances_)[::-1][:SELECT_K]
    Xs = X[:, idx].toarray()
    chosen = {"Random Forest": make_model("Random Forest", best.get("Random Forest")),
              "XGBoost": make_model("XGBoost", best.get("XGBoost")),
              PROPOSED: proposed_model(best)}
    skf = StratifiedKFold(10, shuffle=True, random_state=RNG)
    per_fold_F1, per_fold_MCC, summary = {}, {}, []
    for name, clf in chosen.items():
        pipe = cv_pipe(clf)
        f1s, mf1, roc, pra, mcc = [], [], [], [], []
        for tr, te in skf.split(Xs, y):
            pipe.fit(Xs[tr], y[tr]); yh = pipe.predict(Xs[te]); sc = scores_of(pipe, Xs[te])
            f1s.append(f1_score(y[te], yh)); mf1.append(f1_score(y[te], yh, average="macro"))
            roc.append(roc_auc_score(y[te], sc)); pra.append(average_precision_score(y[te], sc))
            mcc.append(matthews_corrcoef(y[te], yh))
        per_fold_F1[name] = np.array(f1s); per_fold_MCC[name] = np.array(mcc)
        summary.append({"Model": name, "F1": f"{np.mean(f1s):.3f} ± {np.std(f1s):.3f}",
                        "Macro-F1": f"{np.mean(mf1):.3f} ± {np.std(mf1):.3f}",
                        "ROC-AUC": f"{np.mean(roc):.3f} ± {np.std(roc):.3f}",
                        "PR-AUC": f"{np.mean(pra):.3f} ± {np.std(pra):.3f}",
                        "MCC": f"{np.mean(mcc):.3f} ± {np.std(mcc):.3f}"})
        print(f"  CV {name:16s} F1={np.mean(f1s):.3f} PR-AUC={np.mean(pra):.3f}")
    pd.DataFrame(summary).to_csv(os.path.join(OUT, "tab_cv.csv"), index=False)
    pd.DataFrame({n: per_fold_F1[n] for n in chosen}).to_csv(
        os.path.join(OUT, "viral_cv_folds.csv"), index=False)

    sig = []
    for metric, pf in [("F1", per_fold_F1), ("MCC", per_fold_MCC)]:
        a = pf[PROPOSED]
        for basen in ["Random Forest", "XGBoost"]:
            b = pf[basen]; diff = a - b
            tp = stats.ttest_rel(a, b).pvalue; wp = stats.wilcoxon(a, b).pvalue
            sig.append({"Comparison": f"{PROPOSED} vs {basen}", "Metric": metric,
                        "t-test p": round(tp, 4), "Wilcoxon p": round(wp, 4),
                        "Mean diff": round(diff.mean(), 4),
                        "Cohen d": round(diff.mean() / (diff.std(ddof=1) + 1e-12), 3),
                        "Significant (p<0.05)": "Yes" if min(tp, wp) < 0.05 else "No"})
    pd.DataFrame(sig).to_csv(os.path.join(OUT, "tab_significance.csv"), index=False)


def run_ablation(X, y, best):
    """Table VI: five-stage ablation (RF probe S0-S3; S4 = tuned proposed VSE)."""
    nt = TFIDF_FEATURES
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RNG)

    def ev(Xa_tr, Xa_te, clf, resample):
        sc = StandardScaler(with_mean=False); a = sc.fit_transform(Xa_tr); b = sc.transform(Xa_te)
        yy = ytr
        if resample:
            a, yy = SMOTETomek(random_state=RNG).fit_resample(a, ytr)
        clf.fit(a, yy); yh = clf.predict(b)
        return {"Accuracy": accuracy_score(yte, yh), "Recall": recall_score(yte, yh),
                "F1": f1_score(yte, yh), "MCC": matthews_corrcoef(yte, yh)}

    rf = lambda: make_model("Random Forest", best.get("Random Forest"))
    ranker = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=RNG).fit(Xtr, ytr)
    idx = np.argsort(ranker.feature_importances_)[::-1][:SELECT_K]
    rows = [("S0", "TF-IDF text only + RF", ev(Xtr[:, :nt], Xte[:, :nt], rf(), False)),
            ("S1", "+ structured features", ev(Xtr, Xte, rf(), False)),
            ("S2", "+ RF feature selection", ev(Xtr[:, idx].toarray(), Xte[:, idx].toarray(), rf(), False)),
            ("S3", "+ SMOTE-Tomek resampling", ev(Xtr[:, idx].toarray(), Xte[:, idx].toarray(), rf(), True)),
            ("S4", "+ Stacking (full VSE)", ev(Xtr[:, idx].toarray(), Xte[:, idx].toarray(), proposed_model(best), True))]
    b0 = rows[0][2]["F1"]
    out = [{"Stage": s, "Description": d, "Accuracy": round(m["Accuracy"], 4),
            "Recall": round(m["Recall"], 4), "F1": round(m["F1"], 4), "MCC": round(m["MCC"], 4),
            "dF1 vs S0": f"{(m['F1'] - b0) * 100:+.2f} pp"} for s, d, m in rows]
    for r in out:
        print(f"  {r['Stage']}: {r['Description']:30s} F1={r['F1']:.3f}")
    pd.DataFrame(out).to_csv(os.path.join(OUT, "tab_ablation.csv"), index=False)


def full_desc(full):
    full["viral"] = (full["engagement_total"] >= VIRAL_THRESHOLD).astype(int)
    by_sent = full.groupby("sentiment")["viral"].mean()
    pd.Series({"overall_viral_rate": full["viral"].mean(),
               "viral_negative": by_sent.get("negative"), "viral_neutral": by_sent.get("neutral"),
               "viral_positive": by_sent.get("positive")}).to_csv(os.path.join(OUT, "viral_desc.csv"))
    print("viral rate by sentiment:\n", by_sent.round(4))


def main():
    full, samp = load_sample()
    print("sample:", len(samp), "viral rate:", samp["viral"].mean().round(4))
    X, y, names = build_features(samp)
    print("feature matrix:", X.shape)
    full_desc(full)
    print("\n[Tables II/III] hyperparameter tuning + hold-out ...")
    best = tune(X, y, names)
    print("\n[Table VI] ablation ...")
    run_ablation(X, y, best)
    print("\n[Tables IV/V] 10-fold CV + significance ...")
    run_cv(X, y, best)
    print("\ndone. tables + score files in", OUT)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile swa2124_modelfigs.py
# Figures for the viral-tweet modelling study (reads the CSVs written by
# swa2124_models.py) plus the detailed proposed-pipeline architecture diagram.
# All figures share one design system: colour-blind-aware categorical hues, thin
# marks, recessive grids. The proposed model is always the bold dark-red line.

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle
from sklearn.metrics import roc_curve, precision_recall_curve, auc, average_precision_score

BASE = os.path.dirname(os.path.abspath(__file__))
OUT = os.path.join(BASE, "outputs")

INK, MUTED, GRID = "#1a1a1a", "#6b6b6b", "#e6e6e6"
PROPOSED = "Proposed VSE"
MODEL_ORDER = ["Logistic Regression", "Naive Bayes", "k-NN", "Decision Tree",
               "Random Forest", "Extra Trees", "AdaBoost", "SVM (RBF)", "XGBoost",
               "MLP (Neural Net)", PROPOSED]
SHORT = {"Logistic Regression": "LR", "Naive Bayes": "NB", "k-NN": "kNN",
         "Decision Tree": "DT", "Random Forest": "RF", "Extra Trees": "ET",
         "AdaBoost": "Ada", "SVM (RBF)": "SVM", "XGBoost": "XGB",
         "MLP (Neural Net)": "MLP", PROPOSED: "VSE"}
COLORS = {"Logistic Regression": "#4c72b0", "Naive Bayes": "#dd8452",
          "k-NN": "#55a868", "Decision Tree": "#8172b3", "Random Forest": "#937860",
          "Extra Trees": "#64b5cd", "AdaBoost": "#b07aa1", "SVM (RBF)": "#da8bc3",
          "XGBoost": "#8c8c8c", "MLP (Neural Net)": "#ccb974", PROPOSED: "#c1272d"}


def _style():
    plt.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 8, "text.color": INK,
        "axes.edgecolor": "#bdbdbd", "axes.linewidth": 0.8, "axes.labelcolor": INK,
        "axes.labelsize": 8.5, "axes.titlesize": 9, "axes.titleweight": "bold",
        "axes.grid": True, "axes.axisbelow": True,
        "grid.color": GRID, "grid.linewidth": 0.7,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
        "legend.fontsize": 6.6, "legend.frameon": False,
        "figure.dpi": 300, "savefig.dpi": 300,
        "savefig.bbox": "tight", "savefig.pad_inches": 0.03,
    })


def _despine(ax, keep=("left", "bottom")):
    for s in ("top", "right", "left", "bottom"):
        ax.spines[s].set_visible(s in keep)


def lw_of(m):
    return 2.6 if m == PROPOSED else 1.3


def architecture():
    """Detailed end-to-end pipeline diagram with an explicit train/test split
    branch, per-stage data annotations, parallel base learners and a legend."""
    from matplotlib.patches import Patch
    fig, ax = plt.subplots(figsize=(9.4, 9.4))
    ax.set_xlim(0, 100); ax.set_ylim(-5, 106); ax.axis("off")

    C_DATA, C_FEAT, C_SPLIT = "#3a6ea5", "#2e8b57", "#7b5ea7"
    C_TRAIN, C_MODEL, C_TEST, C_EVAL = "#c98a1a", "#c1272d", "#6b6b6b", "#1a7f8c"

    def box(cx, cy, w, h, n, title, sub, color, fill="#f7f9fb", tsize=7.4, ssize=6.4):
        x0, y0 = cx - w / 2, cy - h / 2
        ax.add_patch(FancyBboxPatch((x0, y0), w, h,
                     boxstyle="round,pad=0.15,rounding_size=1.1", fc=fill, ec=color, lw=1.6))
        # coloured header strip
        ax.add_patch(FancyBboxPatch((x0, cy + h / 2 - 3.4), w, 3.4,
                     boxstyle="round,pad=0.15,rounding_size=1.1", fc=color, ec=color, lw=1.0))
        ax.add_patch(plt.Rectangle((x0, cy + h / 2 - 3.4), w, 1.7, fc=color, ec="none"))
        if n:
            ax.add_patch(Circle((x0 + 2.6, cy + h / 2 - 1.7), 1.35, fc="white", ec="none", zorder=5))
            ax.text(x0 + 2.6, cy + h / 2 - 1.7, str(n), ha="center", va="center",
                    color=color, fontsize=6.6, fontweight="bold", zorder=6)
        ax.text(cx + (1.2 if n else 0), cy + h / 2 - 1.7, title, ha="center", va="center",
                color="white", fontsize=tsize, fontweight="bold")
        ax.text(cx, cy - 1.0, sub, ha="center", va="center", color="#2b2b2b",
                fontsize=ssize, linespacing=1.3)
        return {"cx": cx, "cy": cy, "t": cy + h / 2, "b": cy - h / 2,
                "l": cx - w / 2, "r": cx + w / 2}

    def arrow(p1, p2, color="#5a5a5a", lw=1.9, ms=13, style="-|>"):
        ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle=style, mutation_scale=ms,
                     color=color, lw=lw, zorder=1,
                     connectionstyle="arc3,rad=0"))

    def elbow(p1, p2, color="#5a5a5a", lw=1.9, ms=13):
        ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle="-|>", mutation_scale=ms,
                     color=color, lw=lw, zorder=1,
                     connectionstyle="angle,angleA=0,angleB=90,rad=4"))

    # ---- shared preprocessing (top row, left -> right) ----
    yA = 100
    s = box(12.5, yA, 21, 10, None, "Data Source",
            "Harvard Dataverse\ntweets_ai (NHLEJL)\n893,076 tweets", C_DATA, fill="#e9f0f7")
    b1 = box(37.5, yA, 21, 10, 1, "Data Cleaning",
             "English filter + dedupe\nclean URLs / mentions\n820,951 tweets", C_DATA)
    b2 = box(62.5, yA, 21, 10, 2, "Feature Engineering",
             "TF-IDF 1-2g (4,000)\n+ 13 struct. + VADER\n4,013 features", C_FEAT)
    b3 = box(87, yA, 21, 10, 3, "Target Labelling",
             "viral = likes+RT+reply\n>= 6  -  top 11.1%\n(imbalanced)", C_FEAT)
    arrow((s["r"], yA), (b1["l"], yA)); arrow((b1["r"], yA), (b2["l"], yA))
    arrow((b2["r"], yA), (b3["l"], yA))

    # ---- split (centre) ----
    yS = 85
    sp = box(50, yS, 40, 8, 4, "Stratified 80/20 Split",
             "sample n = 12,000  -  seed = 42   ->   train 9,600  |  test 2,400", C_SPLIT,
             fill="#f0ebf6", tsize=7.6, ssize=6.4)
    ax.add_patch(FancyArrowPatch((b3["cx"], b3["b"]), (sp["r"] - 4, sp["t"]),
                 arrowstyle="-|>", mutation_scale=13, color="#5a5a5a", lw=1.9,
                 connectionstyle="arc3,rad=0.0"))

    # ---- TRAIN branch (left column) ----
    xT = 24
    b5 = box(xT, 72, 30, 8, 5, "Z-Score Standardisation", "fit on train only", C_TRAIN, fill="#fdf4e6")
    b6 = box(xT, 60, 30, 8, 6, "RF-Ranked Feature Selection", "rank by RF Gini - keep Top-300", C_TRAIN, fill="#fdf4e6")
    b7 = box(xT, 47, 30, 9, 7, "SMOTE-Tomek Resampling", "oversample + clean overlaps\n-> 17,014 balanced (50/50)", C_TRAIN, fill="#fdf4e6")
    elbow((sp["l"], sp["cy"]), (b5["cx"], b5["t"]))
    arrow((b5["cx"], b5["b"]), (b6["cx"], b6["t"]))
    arrow((b6["cx"], b6["b"]), (b7["cx"], b7["t"]))
    # base learners container with 8 chips
    bl = box(xT, 31, 32, 13, 8, "Base Learners", "", C_MODEL, fill="#fbeceb")
    arrow((b7["cx"], b7["b"]), (bl["cx"], bl["t"]))
    chips = ["LR", "NB", "kNN", "DT", "RF", "SVM", "XGB", "MLP"]
    for i, ch in enumerate(chips):
        cx = bl["l"] + 4.4 + (i % 4) * 6.7
        cy = bl["cy"] - 2.2 - (i // 4) * 4.2
        used = ch in ("LR", "RF", "XGB")
        ax.add_patch(FancyBboxPatch((cx - 2.9, cy - 1.5), 5.8, 3.0,
                     boxstyle="round,pad=0.1,rounding_size=0.6",
                     fc="#c1272d" if used else "white", ec="#c1272d", lw=1.0))
        ax.text(cx, cy, ch, ha="center", va="center", fontsize=5.8, fontweight="bold",
                color="white" if used else "#c1272d")
    vse = box(xT, 16, 32, 9, 9, "Proposed VSE (Stacking)",
              "LR + RF + XGB base learners\n-> Logistic-Regression meta", C_MODEL,
              fill="#f6d9d6", tsize=7.0)
    arrow((bl["cx"], bl["b"]), (vse["cx"], vse["t"]), color="#c1272d", lw=2.2)

    # ---- TEST branch (right column) ----
    xTe = 74
    tf = box(xTe, 72, 30, 8, None, "Held-out Test Fold",
             "2,400 tweets (untouched)", C_TEST, fill="#eeeeee")
    elbow((sp["r"], sp["cy"]), (tf["cx"], tf["t"]))
    ap = box(xTe, 58, 30, 10, None, "Apply Fitted Transforms",
             "same scaler + Top-300\nselection - NO resampling", C_TEST, fill="#eeeeee")
    arrow((tf["cx"], tf["b"]), (ap["cx"], ap["t"]), color="#6b6b6b")

    # ---- convergence: evaluation + validation (bottom row) ----
    ev = box(34, 4, 40, 9, 10, "Evaluation on Test Fold",
             "9 metrics - ROC / PR - confusion - threshold", C_EVAL, fill="#e6f2ec", ssize=6.0)
    va = box(80, 4, 32, 9, 11, "Validation",
             "10-fold CV - Wilcoxon /\nt-tests - ablation study", C_EVAL, fill="#e6f2ec")
    arrow((vse["cx"], vse["b"]), (ev["cx"] - 8, ev["t"]), color="#c1272d", lw=2.1)
    # held-out features feed evaluation (down the right, curving left)
    ax.add_patch(FancyArrowPatch((ap["cx"], ap["b"]), (ev["cx"] + 10, ev["t"]),
                 arrowstyle="-|>", mutation_scale=13, color="#6b6b6b", lw=1.9,
                 connectionstyle="arc3,rad=0.25"))
    arrow((ev["r"], ev["cy"]), (va["l"], va["cy"]), color="#2f7d4f")

    # ---- legend (right-middle empty strip) ----
    handles = [Patch(fc=C_DATA, label="Data acquisition"),
               Patch(fc=C_FEAT, label="Feature engineering"),
               Patch(fc=C_SPLIT, label="Train / test split"),
               Patch(fc=C_TRAIN, label="Imbalance pipeline (train)"),
               Patch(fc=C_MODEL, label="Models / proposed VSE"),
               Patch(fc=C_TEST, label="Held-out test path"),
               Patch(fc=C_EVAL, label="Evaluation & validation")]
    leg = ax.legend(handles=handles, loc="center", ncol=1, fontsize=7.0,
                    frameon=True, bbox_to_anchor=(0.80, 0.34), handlelength=1.1,
                    title="Legend", title_fontsize=7.5)
    leg.get_frame().set_edgecolor("#bbbbbb")
    ax.text(bl["r"] + 1.0, bl["cy"] + 2, "red chips =\nensemble\nbase learners",
            ha="left", va="center", fontsize=5.6, color="#c1272d", fontweight="bold")
    fig.savefig(os.path.join(OUT, "fig_architecture.png"))
    plt.close(fig)


def roc_pr():
    sc = pd.read_csv(os.path.join(OUT, "viral_scores.csv"))
    yt = sc["y_true"].values
    # ROC
    fig, ax = plt.subplots(figsize=(3.5, 3.2))
    ax.plot([0, 1], [0, 1], color="#bcbcbc", lw=1.0, ls="--", zorder=1)
    order = sorted(MODEL_ORDER, key=lambda m: roc_auc(yt, sc[m].values))
    for m in order:
        fpr, tpr, _ = roc_curve(yt, sc[m].values)
        ax.plot(fpr, tpr, color=COLORS[m], lw=lw_of(m),
                label=f"{SHORT[m]} ({auc(fpr, tpr):.3f})",
                zorder=3 if m == PROPOSED else 2)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.legend(loc="lower right", ncol=1)
    _despine(ax)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_roc9.png")); plt.close(fig)
    # PR
    base = yt.mean()
    fig, ax = plt.subplots(figsize=(3.5, 3.2))
    ax.axhline(base, color="#bcbcbc", lw=1.0, ls="--")
    order = sorted(MODEL_ORDER, key=lambda m: average_precision_score(yt, sc[m].values))
    for m in order:
        pr, rc, _ = precision_recall_curve(yt, sc[m].values)
        ax.plot(rc, pr, color=COLORS[m], lw=lw_of(m),
                label=f"{SHORT[m]} ({average_precision_score(yt, sc[m].values):.3f})",
                zorder=3 if m == PROPOSED else 2)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.legend(loc="upper right", ncol=1)
    _despine(ax)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_pr9.png")); plt.close(fig)


def roc_auc(y, s):
    from sklearn.metrics import roc_auc_score
    return roc_auc_score(y, s)


def metric_bars():
    tab = pd.read_csv(os.path.join(OUT, "tab_holdout.csv")).set_index("Model").reindex(MODEL_ORDER)
    mets = ["F1", "Macro-F1", "PR-AUC", "MCC"]
    x = np.arange(len(tab)); w = 0.2
    palette = ["#4c72b0", "#dd8452", "#55a868", "#c44e52"]
    fig, ax = plt.subplots(figsize=(7.0, 2.9))
    for i, m in enumerate(mets):
        ax.bar(x + (i - 1.5) * w, tab[m], w, label=m, color=palette[i])
    ax.set_xticks(x); ax.set_xticklabels([SHORT[m] for m in tab.index])
    ax.set_ylabel("Score"); ax.set_ylim(0, max(0.9, tab[mets].values.max() + 0.08))
    ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.14), columnspacing=1.4)
    ax.axvspan(x[-1] - 0.5, x[-1] + 0.5, color="#c1272d", alpha=0.06)
    _despine(ax); ax.grid(axis="x", visible=False)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_metric_bars.png")); plt.close(fig)


def cv_stability():
    df = pd.read_csv(os.path.join(OUT, "viral_cv_folds.csv"))
    folds = np.arange(1, len(df) + 1)
    fig, ax = plt.subplots(figsize=(3.5, 2.6))
    for m in df.columns:
        ax.plot(folds, df[m], marker="o", ms=4, lw=lw_of(m),
                color=COLORS.get(m, "#333"), label=SHORT.get(m, m))
    ax.set_xlabel("CV fold"); ax.set_ylabel("F1 score")
    ax.set_xticks(folds)
    ax.legend(loc="lower right", ncol=3, columnspacing=1.0)
    _despine(ax)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_cv_stability.png")); plt.close(fig)


def importance():
    imp = pd.read_csv(os.path.join(OUT, "viral_importance.csv"), index_col=0).iloc[:, 0]
    imp = imp.sort_values()
    fig, ax = plt.subplots(figsize=(3.5, 3.4))
    ax.barh(range(len(imp)), imp.values, color="#4c72b0", height=0.72)
    ax.set_yticks(range(len(imp)))
    ax.set_yticklabels([str(i)[:20] for i in imp.index], fontsize=7)
    ax.set_xlabel("Random-forest Gini importance")
    _despine(ax, keep=("bottom",)); ax.tick_params(axis="y", length=0)
    ax.grid(axis="y", visible=False)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_importance.png")); plt.close(fig)


def threshold_sensitivity():
    sc = pd.read_csv(os.path.join(OUT, "viral_scores.csv"))
    yt = sc["y_true"].values
    p = sc[PROPOSED].values
    ths = np.linspace(0.05, 0.95, 91)
    from sklearn.metrics import precision_score, recall_score, f1_score
    P = [precision_score(yt, p >= t, zero_division=0) for t in ths]
    R = [recall_score(yt, p >= t) for t in ths]
    F = [f1_score(yt, p >= t) for t in ths]
    best = ths[int(np.argmax(F))]
    fig, ax = plt.subplots(figsize=(3.5, 2.6))
    ax.plot(ths, P, color="#4c72b0", lw=1.6, label="Precision")
    ax.plot(ths, R, color="#dd8452", lw=1.6, label="Recall")
    ax.plot(ths, F, color="#c1272d", lw=2.2, label="F1")
    ax.axvline(0.5, color="#bcbcbc", lw=0.9, ls="--")
    ax.axvline(best, color="#55a868", lw=1.2, ls=":")
    ax.text(best + 0.01, 0.05, f"F1* @ {best:.2f}", color="#2f7d4f", fontsize=6.6)
    ax.set_xlabel("Decision threshold"); ax.set_ylabel("Score"); ax.set_ylim(0, 1)
    ax.legend(loc="upper right", ncol=3, columnspacing=1.0)
    _despine(ax)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_threshold.png")); plt.close(fig)


def radar():
    tab = pd.read_csv(os.path.join(OUT, "tab_holdout.csv")).set_index("Model")
    mets = ["Accuracy", "Recall", "F1", "ROC-AUC", "PR-AUC", "MCC"]
    show = ["Random Forest", "XGBoost", PROPOSED]
    ang = np.linspace(0, 2 * np.pi, len(mets), endpoint=False)
    ang = np.concatenate([ang, ang[:1]])
    fig, ax = plt.subplots(figsize=(3.4, 3.2), subplot_kw=dict(polar=True))
    for m in show:
        v = tab.loc[m, mets].values.astype(float)
        v = np.concatenate([v, v[:1]])
        ax.plot(ang, v, color=COLORS[m], lw=lw_of(m), label=SHORT[m])
        ax.fill(ang, v, color=COLORS[m], alpha=0.08)
    ax.set_xticks(ang[:-1]); ax.set_xticklabels(mets, fontsize=7)
    ax.set_ylim(0, 1); ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.50", "0.75", "1.0"], fontsize=6, color=MUTED)
    ax.grid(color=GRID)
    ax.legend(loc="lower right", bbox_to_anchor=(1.15, -0.08))
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_radar.png")); plt.close(fig)


def confusion():
    cm = pd.read_csv(os.path.join(OUT, "viral_confusion.csv")).values.astype(int)
    cmn = cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(3.0, 2.7))
    ax.imshow(cmn, cmap="Reds", vmin=0, vmax=1)
    labs = ["Non-viral", "Viral"]
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}\n{cmn[i, j]*100:.1f}%", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.5 else INK, fontsize=8.5, fontweight="bold")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(labs); ax.set_yticklabels(labs, rotation=90, va="center")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_xticks(np.arange(-.5, 2, 1), minor=True)
    ax.set_yticks(np.arange(-.5, 2, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.grid(which="major", visible=False); ax.tick_params(which="minor", length=0)
    _despine(ax, keep=())
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_confusion_viral.png")); plt.close(fig)


def viral_descriptive():
    df = pd.read_csv(os.path.join(OUT, "processed_tweets.csv"),
                     usecols=["sentiment", "has_photo", "has_video", "engagement_total"])
    df["viral"] = (df["engagement_total"] >= 6).astype(int)
    order = ["negative", "neutral", "positive"]
    sc = {"positive": "#009E73", "neutral": "#9a9a9a", "negative": "#D55E00"}
    g = df.groupby("sentiment")["viral"].mean().reindex(order) * 100
    # the dataset only contains three media combinations: (0,0), (1,1), (0,1)
    combo = {(0, 0): "Text only", (1, 1): "Photo", (0, 1): "Video / GIF"}
    df["media"] = [combo.get((p, v), "other") for p, v in zip(df.has_photo, df.has_video)]
    gm = df.groupby("media")["viral"].mean().reindex(
        ["Text only", "Photo", "Video / GIF"]) * 100
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.0, 2.5))
    b1 = a1.bar(order, g.values, color=[sc[s] for s in order], width=0.62)
    a1.bar_label(b1, labels=[f"{v:.1f}" for v in g.values], padding=2, fontsize=7)
    a1.set_title("Viral rate by sentiment", loc="left", fontsize=8.5)
    a1.set_ylabel("Viral tweets (%)"); a1.set_ylim(0, max(g.values) * 1.25)
    a1.set_xticks(range(3)); a1.set_xticklabels(order, rotation=15, ha="right")
    b2 = a2.bar(gm.index, gm.values, color=["#8c8c8c", "#4c72b0", "#c1272d"], width=0.62)
    a2.bar_label(b2, labels=[f"{v:.1f}" for v in gm.values], padding=2, fontsize=7)
    a2.set_title("Viral rate by media type", loc="left", fontsize=8.5)
    a2.set_ylabel("Viral tweets (%)"); a2.set_ylim(0, max(gm.values) * 1.25)
    for a in (a1, a2):
        _despine(a); a.grid(axis="x", visible=False)
    fig.tight_layout(w_pad=1.5)
    fig.savefig(os.path.join(OUT, "fig_viral_descriptive.png")); plt.close(fig)


def tuning_impact():
    """Default vs tuned F1 for every baseline (impact of hyperparameter search)."""
    t = pd.read_csv(os.path.join(OUT, "tab_tuning.csv"))
    order = [m for m in MODEL_ORDER if m in set(t["Model"])]
    t = t.set_index("Model").reindex(order)
    x = np.arange(len(t)); w = 0.38
    fig, ax = plt.subplots(figsize=(7.0, 2.7))
    ax.bar(x - w / 2, t["F1 (default)"], w, label="Default params", color="#b8b8b8")
    ax.bar(x + w / 2, t["F1 (tuned)"], w, label="Tuned params", color="#4c72b0")
    ax.set_xticks(x); ax.set_xticklabels([SHORT[m] for m in t.index])
    ax.set_ylabel("F1 score"); ax.set_ylim(0, max(t["F1 (tuned)"].max(), 0.35) + 0.06)
    ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.15))
    _despine(ax); ax.grid(axis="x", visible=False)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_tuning_impact.png")); plt.close(fig)


def metric_heatmap():
    """Heatmap of every model across the nine metrics (column-normalised shading)."""
    tab = pd.read_csv(os.path.join(OUT, "tab_holdout.csv")).set_index("Model").reindex(MODEL_ORDER)
    mets = ["Accuracy", "Precision", "Recall", "F1", "Macro-F1", "ROC-AUC", "PR-AUC", "MCC", "Kappa"]
    M = tab[mets].values.astype(float)
    # normalise each column to [0,1] so shading compares models within a metric
    norm = (M - M.min(axis=0)) / (np.ptp(M, axis=0) + 1e-9)
    fig, ax = plt.subplots(figsize=(7.0, 3.6))
    ax.imshow(norm, cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(mets))); ax.set_xticklabels(mets, rotation=30, ha="right", fontsize=7)
    ax.set_yticks(range(len(tab))); ax.set_yticklabels([SHORT[m] for m in tab.index], fontsize=7.5)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=6.2,
                    color="white" if norm[i, j] > 0.6 else "#222")
    # highlight the proposed row
    pi = list(tab.index).index(PROPOSED)
    ax.add_patch(plt.Rectangle((-0.5, pi - 0.5), len(mets), 1, fill=False,
                 edgecolor="#c1272d", lw=2.0))
    ax.set_xticks(np.arange(-.5, len(mets), 1), minor=True)
    ax.set_yticks(np.arange(-.5, len(tab), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.0); ax.tick_params(which="minor", length=0)
    ax.grid(which="major", visible=False); _despine(ax, keep=())
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_metric_heatmap.png")); plt.close(fig)


def monthly_volume():
    """Fig 2: monthly tweet volume (top) and mean VADER sentiment (bottom)."""
    df = pd.read_csv(os.path.join(OUT, "processed_tweets.csv"),
                     usecols=["month", "id", "vader_compound"])
    m = df.groupby("month").agg(vol=("id", "count"), sent=("vader_compound", "mean"))
    xi = np.arange(len(m))
    tick = [i for i, v in enumerate(m.index) if v.endswith("-01")]
    lab = [m.index[i][:4] for i in tick]
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(3.5, 3.1), sharex=True,
                                 gridspec_kw={"height_ratios": [1.2, 1]})
    a1.fill_between(xi, m["vol"], color="#0072B2", alpha=0.18)
    a1.plot(xi, m["vol"], color="#0072B2", lw=1.8)
    a1.set_ylabel("Tweets / month"); a1.set_title("Tweet volume", loc="left", fontsize=8.5)
    a2.axhline(0.05, color="#c9c9c9", lw=0.7, ls="--")
    a2.plot(xi, m["sent"], color="#009E73", lw=1.8)
    a2.set_ylabel("Mean VADER"); a2.set_title("Average sentiment", loc="left", fontsize=8.5)
    a2.set_xticks(tick); a2.set_xticklabels(lab); a2.set_xlabel("Year")
    for a in (a1, a2):
        _despine(a); a.grid(axis="x", visible=False)
    fig.tight_layout(h_pad=0.8); fig.savefig(os.path.join(OUT, "fig_monthly_volume.png")); plt.close(fig)


def sentiment_area():
    """Fig 3: yearly sentiment composition as a 100% stacked area."""
    from matplotlib.ticker import PercentFormatter
    sc = {"positive": "#009E73", "neutral": "#9a9a9a", "negative": "#D55E00"}
    df = pd.read_csv(os.path.join(OUT, "processed_tweets.csv"), usecols=["year", "sentiment"])
    sy = df.groupby(["year", "sentiment"]).size().unstack().fillna(0)
    share = sy.div(sy.sum(axis=1), axis=0) * 100
    order = ["negative", "neutral", "positive"]
    fig, ax = plt.subplots(figsize=(3.5, 2.5))
    ax.stackplot(share.index, [share[s] for s in order], colors=[sc[s] for s in order],
                 edgecolor="white", linewidth=1.2)
    ax.set_ylim(0, 100); ax.yaxis.set_major_formatter(PercentFormatter())
    ax.set_xticks(share.index); ax.set_xlabel("Year"); ax.set_ylabel("Share of tweets")
    xr = share.index.max(); cum = 0
    for s in order:
        v = share[s].iloc[-1]
        ax.text(xr + 0.06, cum + v / 2, s, va="center", ha="left", color=sc[s],
                fontsize=7.5, fontweight="bold"); cum += v
    ax.set_xlim(share.index.min(), xr + 0.9)
    _despine(ax); ax.grid(axis="x", visible=False)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_sentiment_by_year.png")); plt.close(fig)


def day_hour_heatmap():
    """Fig 5: engagement rate (%) by day of week and hour of day."""
    df = pd.read_csv(os.path.join(OUT, "processed_tweets.csv"),
                     usecols=["dayofweek", "hour", "engaged"])
    piv = (df.groupby(["dayofweek", "hour"])["engaged"].mean().unstack() * 100).reindex(range(7))
    fig, ax = plt.subplots(figsize=(7.0, 2.6))
    im = ax.imshow(piv.values, aspect="auto", cmap="Blues",
                   vmin=np.nanpercentile(piv.values, 5), vmax=np.nanpercentile(piv.values, 95))
    ax.set_yticks(range(7)); ax.set_yticklabels(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
    ax.set_xticks(range(0, 24, 2)); ax.set_xticklabels(range(0, 24, 2))
    ax.set_xlabel("Hour of day (EST)"); ax.set_ylabel("Day of week")
    ax.set_xticks(np.arange(-.5, 24, 1), minor=True); ax.set_yticks(np.arange(-.5, 7, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.0); ax.grid(which="major", visible=False)
    ax.tick_params(which="minor", length=0)
    cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cb.set_label("Engaged (%)", fontsize=7.5); cb.ax.tick_params(labelsize=7)
    _despine(ax, keep=())
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig_heatmap_day_hour.png")); plt.close(fig)


def main():
    _style()
    architecture()
    monthly_volume()
    sentiment_area()
    day_hour_heatmap()
    viral_descriptive()
    roc_pr()
    metric_bars()
    metric_heatmap()
    tuning_impact()
    cv_stability()
    importance()
    threshold_sensitivity()
    radar()
    confusion()
    print("model figures saved to", OUT)


if __name__ == "__main__":
    main()


## 4. Run the whole pipeline
Step 1 cleans the data, Step 2 trains + tunes the 11 models and writes all result tables, Step 3 draws every figure, Step 4 writes the Power BI data files.

In [ ]:
!python swa2124_analysis.py preprocess    # ~2 min

In [ ]:
!python swa2124_models.py                 # ~15 min (11 models + tuning + CV)

In [ ]:
!python swa2124_modelfigs.py              # all figures

In [ ]:
!python swa2124_analysis.py exports       # Power BI data files

## 5. Every result table (Tables III–VIII)

In [ ]:
import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_columns", 30)

tables = [("Table III — Per-algorithm hyperparameter tuning", "outputs/tab_tuning.csv"),
          ("Table IV — Full hold-out performance (11 models x 9 metrics)", "outputs/tab_holdout.csv"),
          ("Table V — 10-fold cross-validation (mean +/- std)", "outputs/tab_cv.csv"),
          ("Table VI — Paired significance tests", "outputs/tab_significance.csv"),
          ("Table VII — Five-stage ablation study", "outputs/tab_ablation.csv")]
for title, path in tables:
    display(Markdown(f"### {title}")); display(pd.read_csv(path))

# Table VIII — comparison with existing studies (proposed row pulled from the results above)
vse = pd.read_csv("outputs/tab_holdout.csv").set_index("Model").loc["Proposed VSE"]
comparison = pd.DataFrame([
 ["Rustam et al. [4]", "Extra Trees / XGBoost / LSTM", "7,528 COVID-19 tweets (sentiment)", "Acc 0.93 (extra trees)"],
 ["Chakraborty et al. [5]", "Fuzzy rule base + classical", "226,668 COVID-19 tweets (sentiment)", "Acc up to 0.81"],
 ["Naseem et al. [6]", "Classical vs deep vs transformer", "90,000 tweets (COVIDSenti, sentiment)", "Transformer models best"],
 ["Vohra & Garg [7]", "CNN + FastText", "358,823 WFH tweets (sentiment)", "Acc 0.926"],
 ["Alharbi & de Doncker [9]", "CNN + user behaviour", "SemEval Twitter (sentiment)", "Beats text-only baselines"],
 ["Andariesta & Wasesa [10]", "LR / DT / KNN / RF", "12,786 e-commerce tweets (engagement)", "4-tier engagement classification"],
 ["Mestrovic et al. [11]", "BERT + multilayer network", "199,431 tweets (retweet-count classes)", "Text + network best"],
 ["Proposed VSE (this study)", "Resampled stacking ensemble", "893,076 AI tweets (virality, imbalanced)",
  f"F1 {vse['F1']:.3f}, PR-AUC {vse['PR-AUC']:.3f}, MCC {vse['MCC']:.3f}, ROC-AUC {vse['ROC-AUC']:.3f}"],
], columns=["Existing study", "Model", "Dataset (task)", "Reported performance"]).set_index("Existing study")
display(Markdown("### Table VIII — Comparison of the proposed model with existing studies"))
display(comparison)

## 6. Every figure / diagram (Figs 1–12 and appendix)
All images below are the exact figures embedded in the report.

In [ ]:
import os
from IPython.display import Image, display, Markdown
figs = [
 ("fig_architecture",      "Fig. 1  — Research framework / architecture"),
 ("fig_monthly_volume",    "Fig. 2  — Tweet volume and mean sentiment over time"),
 ("fig_sentiment_by_year", "Fig. 3  — Yearly sentiment composition"),
 ("fig_viral_descriptive", "Fig. 4  — Viral rate by sentiment and by media type"),
 ("fig_heatmap_day_hour",  "Fig. 5  — Engagement rate by day and hour"),
 ("fig_tuning_impact",     "Fig. 6  — Default vs tuned F1 per model"),
 ("fig_roc9",              "Fig. 7  — ROC curves (11 models)"),
 ("fig_pr9",               "Fig. 8  — Precision-recall curves (11 models)"),
 ("fig_metric_bars",       "Fig. 9  — F1 / macro-F1 / PR-AUC / MCC per model"),
 ("fig_metric_heatmap",    "Fig. 10 — Heatmap of all models x all metrics"),
 ("fig_confusion_viral",   "Fig. 11 — Confusion matrix of the proposed VSE"),
 ("fig_importance",        "Fig. 12 — Top-15 features by RF importance"),
 ("fig_cv_stability",      "Fig. A1 — Per-fold F1 across 10 CV folds"),
 ("fig_threshold",         "Fig. A2 — Threshold sensitivity of the proposed VSE"),
 ("fig_radar",             "Fig. A3 — Radar comparison across six metrics"),
]
for base, caption in figs:
    p = f"outputs/{base}.png"
    if os.path.exists(p):
        display(Markdown(f"**{caption}**")); display(Image(p))

## 7. Notes
- Everything above is produced by the code in Section 3 — nothing is hard-coded.
- The Power BI dashboard is built separately from the `outputs/pbi_*.csv` files (see the report / instructions).
- **Dataset citation:** N. de Marcellis-Warin, D. Kouloukoui, and T. Warin, "A large-scale dataset of
  AI-related tweets: Structure and descriptive statistics," *Data in Brief*, vol. 62, art. 111960, 2025,
  Harvard Dataverse, DOI: 10.7910/DVN/NHLEJL.
